## SUSAN Corner Detection + Convex Hull Thresholding

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print("✓ All libraries imported successfully!")
print("Note: Only numpy, matplotlib, and PIL used - no CV libraries!")

✓ All libraries imported successfully!
Note: Only numpy, matplotlib, and PIL used - no CV libraries!


### 1. SUSAN Implementation

### 2. Convex Hull Implementation + Two Pass Threshold Implementation

#### 2.a Convex Hull Implementation

In [2]:
def CalculateHullPoints(histogram):
    """
    Calculate convex hull points
    
    Algorithm :
    1. For color intensity i, find intensity j that gives maximum slope
    2. After getting maximum slope at j, let i = j
    3. Repeat until i = 254, making j start from 255
    
    Parameters:
    -----------
    histogram : 1D numpy array
        Histogram of image intensities
        
    Returns:
    --------
    hull_points : list of tuples
        List of (start, end) tuples representing convex hull segments
    """
    n = len(histogram)
    hull_points = []
    
    i = 0
    
    while i < n - 1:
        max_slope = -np.inf
        best_j = i + 1

        for j in range(i + 1, n):  # here i is fixed and j varies from i+1 to 255
            slope = (histogram[j] - histogram[i]) / (j - i)
            
            if slope > max_slope:
                max_slope = slope
                best_j = j
        
        hull_points.append((i, best_j))
        
        i = best_j 
        
        if i >= n - 1:
            break
    
    return hull_points

In [3]:
def GetHullValues(hull_points, histogram):
    """
    Calculate hull values using straight line equation: y = mx + c
    
    Parameters:
    -----------
    hull_points : list of tuples
        List of (start, end) tuples from convex hull
    histogram : 1D numpy array
        Original histogram
    
    Returns:
    --------
    hull_values : 1D numpy array
        Hull values for each intensity calculated using line equation
    """
    n = len(histogram)
    hull_values = np.zeros(n, dtype=np.float32)
    
    for start, end in hull_points:
        if start == end:
            hull_values[start] = histogram[start]
            continue
        
        h_i = histogram[start]  
        h_j = histogram[end]    
        
        m = (h_j - h_i) / (end - start)
        
        c = h_i - m * start
        
        for x in range(start, end + 1):
            hull_values[x] = m * x + c
    
    return hull_values

In [4]:
def GetThreshold(histogram, hull_values):
    """
    Find threshold with maximum difference between hull and histogram
    
    Formula: Thr = Max_{i=0}^{255}(Hull(i) - Hist(i))
    
    Parameters:
    -----------
    histogram : 1D numpy array
        Original histogram
    hull_values : 1D numpy array
        Convex hull values
    
    Returns:
    --------
    threshold : int
        Optimal threshold value (intensity with maximum difference)
    """
    differences = hull_values - histogram
    
    threshold = np.argmax(differences)
    
    return int(threshold)

In [5]:
def ApplyThresh(histogram, threshold):
    """
    Split histogram at threshold
    
    Parameters:
    -----------
    histogram : 1D numpy array
        Input histogram
    threshold : int
        Threshold value to split at
    
    Returns:
    --------
    below_thresh : 1D numpy array
        Histogram values below threshold
    above_thresh : 1D numpy array
        Histogram values above threshold
    """
    n = len(histogram)
    
    below_thresh = np.zeros(n, dtype=histogram.dtype)
    above_thresh = np.zeros(n, dtype=histogram.dtype)
    
    for i in range(n):
        if i <= threshold:
            below_thresh[i] = histogram[i]
        else:
            above_thresh[i] = histogram[i]
    
    return below_thresh, above_thresh

#### 2.b Two Pass Threshold Implementation

In [6]:
def two_pass_convex_hull_threshold(histogram):
    """
    Apply convex hull thresholding twice to get two thresholds
    
    Parameters:
    -----------
    histogram : 1D numpy array
        Input histogram
    
    Returns:
    --------
    threshold1 : int
        First threshold (separates corners from edges)
    threshold2 : int
        Second threshold (separates edges from homogeneous regions)
    """    
    # on the full histogram
    hull_points_1 = CalculateHullPoints(histogram)
    hull_values_1 = GetHullValues(hull_points_1, histogram)
    threshold1 = GetThreshold(histogram, hull_values_1)
    
    print(f"Threshold 1 (corners/edges): {threshold1}")
    
    below_t1, above_t1 = ApplyThresh(histogram, threshold1)
        
    # on the above threshold1 part of histogram
    hull_points_2 = CalculateHullPoints(above_t1)
    hull_values_2 = GetHullValues(hull_points_2, above_t1)
    threshold2 = GetThreshold(above_t1, hull_values_2)
    
    print(f"Threshold 2 (edges/homogeneous): {threshold2}")
    
    if threshold2 <= threshold1:
        threshold2 = threshold1 + max(1, (255 - threshold1) // 3)
        print(f"Adjusted Threshold 2 to: {threshold2}")
    
    return threshold1, threshold2

### 3. Draw Image
